# Hypothesis Testing

Now that the metrics and tracking implementation plan are decided, we must perform a power analysis to derive the required sample size for our hypothesis testing. This can be calculated using our baseline OEC metrics and MDE of 2%. 

## Power Analysis

In [2]:
import math
import statsmodels.stats.power as power

# Assumed bassline OEC metrics and MDE
baseline_mean = 420.0  # Historic baseline minutes per user per week
baseline_std = 180.0  # Assumed behavioral standard deviation
mde_percent = 0.02  # 2% Minimum Detectable Effect

# Calculate the Absolute Detectable Effect and Effect Size (Cohen's d)
absolute_effect = baseline_mean * mde_percent
cohens_d = absolute_effect / baseline_std

print("--- Experiment Design Parameters ---")
print(f"Baseline Weekly Listening: {baseline_mean} minutes")
print(f"Target MDE Lift: {mde_percent * 100}% (+{absolute_effect:.2f} minutes)")
print(f"Calculated Effect Size (Cohen's d): {cohens_d:.4f}\n")

# Solve for sample size per variant (two-sided independent t-test)
analysis = power.TTestIndPower()
required_n_per_variant = analysis.solve_power(
    effect_size=cohens_d, 
    alpha=0.05, 
    power=0.80, 
    ratio=1.0, 
    alternative="two-sided"
)

print("--- Power Analysis Results ---")
print(f"Required sample size per variant: {math.ceil(required_n_per_variant)}")
print(f"Total users enrolled (1:1 Sample Ratio): {math.ceil(required_n_per_variant) * 2}")

--- Experiment Design Parameters ---
Baseline Weekly Listening: 420.0 minutes
Target MDE Lift: 2.0% (+8.40 minutes)
Calculated Effect Size (Cohen's d): 0.0467

--- Power Analysis Results ---
Required sample size per variant: 7210
Total users enrolled (1:1 Sample Ratio): 14420


## Dataset Validation

Next we'll load in the simulated dataset (generated via `data_simulation.py`) to check for Sample Ratio Mismatch (SRM). This will be done using a Chi-Square Goodness-of-Fit test. 

In [5]:
import pandas as pd
import scipy.stats as stats

# 1. Load the generated data file
df = pd.read_csv("data/simulated_experiment_log.csv")

# 2. Extract unique users per variant (Do not count daily rows, count unique people!)
observed_counts = df.groupby("variant")["user_id"].nunique().values
total_observed_users = observed_counts.sum()

# 3. Define our expected 50/50 split ratio
expected_counts = [total_observed_users / 2, total_observed_users / 2]

print("--- Sample Ratio Mismatch (SRM) Validation Check ---")
print(f"Total Unique Users Enrolled: {total_observed_users:,}")
print(f"Observed - Control: {observed_counts[0]:,} | Treatment: {observed_counts[1]:,}")
print(
    f"Expected - Control: {expected_counts[0]:,} | Treatment: {expected_counts[1]:,}\n"
)

# 4. Execute Chi-Square Goodness-of-Fit Test
chi2_stat, p_value = stats.chisquare(f_obs=observed_counts, f_exp=expected_counts)

print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"SRM Validation p-value: {p_value:.6f}")

# 5. Strategic Evaluation Rule
# In industry, an SRM check uses a super strict alpha of 0.001 to flag catastrophic pipeline bugs.
if p_value < 0.001:
    print(
        "🚨 CRITICAL WARNING: Sample Ratio Mismatch detected! The experiment assignment pipeline is corrupted."
    )
else:
    print(
        "✅ PASSED: No Sample Ratio Mismatch detected. Traffic allocation matches expectations."
    )

--- Sample Ratio Mismatch (SRM) Validation Check ---
Total Unique Users Enrolled: 14,420
Observed - Control: 7,301 | Treatment: 7,119
Expected - Control: 7,210.0 | Treatment: 7,210.0

Chi-Square Statistic: 2.2971
SRM Validation p-value: 0.129617
✅ PASSED: No Sample Ratio Mismatch detected. Traffic allocation matches expectations.
